# 36 — BẢN GHI SHOT cho batch 1 (văn bản → văn bản, qua cổng Trammay)\n
Tầng này **không nhìn ảnh**. Nó đọc caption keyframe đã sinh (`v5_bao`, Gemini 3.8) rồi viết cho mỗi shot một bản ghi mang đúng thứ caption keyframe KHÔNG thể có: **chuyện gì xảy ra**, **cái gì đổi**, và **shot này nối với shot trước thế nào**.\n
**Vì sao text→text chứ không gọi lại VLM:** caption `v5_bao` có 49,7% ≥3 gốc động từ (bản Qwen cũ: 5,0%) và dài 185 token. Tiền đề của thiết kế 17-09 — *"caption thiếu diễn biến nên tầng shot phải tự nhìn ảnh"* — không còn đúng. Đo 24-09: **59M token** thay vì **392M**, rẻ hơn 6,6 lần, và không tranh quota với lượt sinh caption batch 2.\n
**Vì sao MỘT CỤM shot mỗi request, thay cho vòng hồi quy từng shot:** ReCap truyền `M_{t-1}` qua từng shot vì VLM chỉ nhìn được ảnh của một shot mỗi lượt. Đầu vào là văn bản thì nhét ~10 shot liền nhau vào một prompt được — model thấy cả chuỗi và tự viết liên tục tường thuật. **8.751 lượt gọi thay vì 87.508**, và các cụm độc lập nhau nên song song được BÊN TRONG một video.\n
⚠ **Chia cụm theo NGÂN SÁCH TOKEN, không theo số shot.** 469 shot (0,48%) ôm 20.224 keyframe = **11,5% cả kho**; `L25_V054#s0065` có **196 keyframe / 1.171 giây**. Chia theo số shot là nhét 36.000 token vào một mục.\n
⚠ **10,3% shot (10.089) không có keyframe nào** nên đường văn bản không chạm tới. Notebook ĐẾM và báo chúng, không im lặng bỏ qua.\n
⚠ **Tầng này CHỈ phủ batch 1.** Batch 2 chưa có ranh giới shot. Đưa arm shot vào hợp nhất lúc này là tạo thiên lệch hai nửa kho — xem `docs/research/thiet-ke-caption-shot-level.md` §10.5.\n
## Trước khi Run All\n
1. Sửa `RUNNER`, đặt `RUN_SCOPE = "SHARD"` và `CONFIRM_SHARD` **bằng đúng** `RUNNER`.
2. Bật 4 secret: `TRAMMAY_API_KEY` · `GITHUB_TOKEN` · `MODAL_TOKEN_ID` · `MODAL_TOKEN_SECRET`.
3. **Không gắn dataset nào.** Caption và gói shot đều kéo từ Modal.
4. Gói audit shot phải có trên volume — đẩy một lần bằng `scratchpad/day_goi_shot_len_volume.py`.\n

In [ ]:
from pathlib import Path

# ── ai chạy, chạy phạm vi nào ──────────────────────────────────────────────
RUNNER = "w1"                     # w1..w6
RUNNERS = ("w1", "w2", "w3", "w4", "w5", "w6")
RUN_SCOPE = "PREFLIGHT"           # PREFLIGHT | SHARD | AGGREGATE
CONFIRM_SHARD = ""                # SHARD: đặt đúng bằng RUNNER
CONFIRM_AGGREGATE = False         # AGGREGATE: True sau khi cả 6 shard COMPLETE

# ── cổng API ───────────────────────────────────────────────────────────────
# Kaggle Secrets: Add-ons -> Secrets -> TRAMMAY_API_KEY
TRAMMAY_SECRET = "TRAMMAY_API_KEY"
MODEL_NAME = "ag/gemini-3.8-flash"
PROMPT_VARIANT = "ban_ghi_shot"   # prompt ở mfe.offline.shot_record_prompt
# Cổng BỎ QUA `max_tokens` (đo 22-09: gửi 40 vẫn trả 263 token, `finish_reason
# ='stop'`), nên con số này chỉ để qua cửa kiểm tra. Trần của endpoint batch là
# 8.192; chat thường vẫn 4.096.
MAX_TOKENS = 8192
# ── đường gọi: "chat" (1 ảnh/1 request) hay "batch" (k ảnh/1 request) ───────
#
# Chọn theo thứ đang CHẶN, không theo thứ nghe nhanh hơn:
#
#   "batch"  khi trần REQUEST chặn (10.000/ngày/user). Giảm k lần số request,
#            đổi lại chậm hơn 1,47 lần và caption cần phụ lục `len180` để không
#            tụt từ 195 xuống 148 từ.
#   "chat"   khi trần request đã nới. Nhanh hơn (2.040 so 1.388 khung/giờ/
#            worker) và tái lập ĐÚNG cấu hình đã sinh 499 video đêm 21-09.
#
# ⚠ `BATCH_SIZE = 1` KHÔNG tương đương `"chat"`: nó vẫn đi qua
# `/chat/completions/batch`, tức vẫn dính system prompt ép JSON mà cổng chèn —
# chính thứ làm model tự chia phần. Muốn một-ảnh-một-request thật thì phải là
# `"chat"`.
DUONG_GOI = "chat"                # cụm shot LUÔN đi đường chat
MAX_TOKENS_CHAT = 8192            # ~10 bản ghi JSON trong một lượt
MOC_GHI = 80 if DUONG_GOI == "chat" else 10   # ghi checkpoint mỗi ~80 ảnh
BATCH_SIZE = 8                    # chỉ dùng khi DUONG_GOI == "batch"
BATCH_MAX_BYTES = 7_000_000       # hàng rào THẬT: cắt theo byte — xem bảng dưới
# Số luồng và nền lùi phải khớp ĐƯỜNG GỌI — cả hai đều theo độ dài một request:
#   "chat"  : ~8s/lượt  -> 28 luồng, lùi 2s   (đo đêm 21-09: 6–9 giờ sạch)
#   "batch" : 23–47s    ->  2 luồng, lùi 30s  (đo 22-09: 0/18 hỏng)
# Đặt lẫn là hỏng theo hai kiểu: 28 luồng batch thì 524 hàng loạt, còn 2 luồng
# chat thì chậm gấp 14 lần mà chẳng an toàn hơn.
MAX_WORKERS = 12                  # cụm nặng hơn một ảnh; 28 là quá tay
BACKOFF_BASE = 2.0 if DUONG_GOI == "chat" else 30.0
REQUEST_TIMEOUT = 300
MAX_RETRIES = 4

# Chạm hạn mức 5 tiếng của thượng nguồn thì NGỦ CHỜ, không thoát: phiên Kaggle
# sống 12 tiếng nên chờ hết một cửa sổ vẫn còn dư thời gian làm việc, còn thoát
# ra là phải có người ngồi canh bấm chạy lại (đo 24-09: một worker dừng lúc
# 4,95 giờ rồi nằm không suốt đêm).
#
# 330 phút = 5,5 tiếng: một cửa sổ đầy cộng biên. Chờ lâu hơn thì Kaggle giết
# phiên trước — mà chuyện đó vô hại, marker và checkpoint đã nằm trên volume.
CHO_QUOTA_PHUT = 330
# Thăm dò bằng một lượt gọi KHÔNG ẢNH. Không ai biết cửa sổ bắt đầu lúc nào, nên
# ngủ cứng 5 tiếng là luôn chọn trường hợp xấu nhất; thăm dò 10 phút một lần tốn
# tối đa ~30 request trên 1.000 của cửa sổ.
NHIP_THAM_DO_PHUT = 10

# ── đường dẫn ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/triet/structure"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/caption-gemini")

# Nguồn ảnh + map-keyframes: volume BTC, KHÔNG phải `/kaggle/input`.
BTC_VOLUME = "keyframes-btc"
BTC_BUNDLE = "aic_btc_b1_v1"
CACHE_ANH = WORK_DIR / "keyframes"

# Gốc RIÊNG. Bản ghi shot là artifact khác caption keyframe — trộn chung
# một gốc là mất khả năng gộp lại hai thứ độc lập về sau.
OUTPUT_ROOT = "/research/shot-records-v1"
CAPTION_ROOT = "/research/caption-gemini-trammay-v1"   # ĐỌC caption keyframe từ đây
SHOT_ZIP_REMOTE = "/research/shot-records-v1/shot-detection-audit-2026-09-17.zip"

# Chia cụm theo NGÂN SÁCH TOKEN, không theo số shot: 469 shot (0,48%) ôm
# 20.224 keyframe (11,5% cả kho), và `L25_V054#s0065` có 196 keyframe /
# 1.171 giây — TransNetV2 không cắt được một trường quay tĩnh. Chia theo
# số shot là nhét 36.000 token vào một mục.
CUM_SHOT_TOI_DA = 10              # trần TRÊN, ngân sách token mới quyết
CUM_TOKEN_TOI_DA = 6000           # ~4 ký tự/token, đo trên caption thật
# Trong một shot dài thì LẤY MẪU THƯA keyframe (đầu · giữa · cuối) chứ
# không nhồi hết: 196 caption của cùng một cú máy tĩnh không nói thêm gì.
MAX_KF_TRONG_SHOT = 8
RESEARCH_VOLUME = "keyframe-augmentation-research"

EXPECTED_VIDEOS = 873
EXPECTED_FRAMES = 176707          # khung, để đối chiếu; shot đếm riêng
EXPECTED_SHOTS = 97597            # gói audit TransNetV2 17-09

if RUNNER not in RUNNERS:
    raise ValueError(f"RUNNER phải thuộc {RUNNERS}")
if RUN_SCOPE not in {"PREFLIGHT", "SHARD", "AGGREGATE"}:
    raise ValueError("RUN_SCOPE phải là PREFLIGHT, SHARD hoặc AGGREGATE")
if RUN_SCOPE == "SHARD" and CONFIRM_SHARD != RUNNER:
    raise RuntimeError("SHARD bị khoá; đặt CONFIRM_SHARD đúng bằng RUNNER")
if RUN_SCOPE == "AGGREGATE" and not CONFIRM_AGGREGATE:
    raise RuntimeError("AGGREGATE bị khoá; đặt CONFIRM_AGGREGATE = True")
if DUONG_GOI not in {"chat", "batch"}:
    raise ValueError('DUONG_GOI phải là "chat" hoặc "batch"')
if not 1 <= BATCH_SIZE <= 8:
    raise ValueError("BATCH_SIZE phải từ 1 đến 8")
if BATCH_MAX_BYTES < 700_000 or BATCH_MAX_BYTES > 7_500_000:
    # Trần thân request của cổng là 8 MB (Đạt nâng 22-09; đo: 5 MB qua, 8 MB
    # trả 413). 7,5 MB là mức cao nhất còn chừa biên cho overhead JSON.
    #
    # ⚠ KÍCH THƯỚC ẢNH KHÁC NHAU RẤT NHIỀU giữa hai batch và giữa các nhóm nội
    # dung — đo 23-09 trên 208.491 keyframe batch 2 (zip / số khung, theo worker):
    #
    #     batch 1              178 KB/ảnh  -> base64 237 KB -> k=8 = 1,85 MiB
    #     batch 2 M tin tức    388 KB      -> 517 KB        -> k=8 = 4,04 MiB
    #     batch 2 S đua xe     393 KB      -> 524 KB        -> k=8 = 4,10 MiB
    #     batch 2 N giao thông 582 KB      -> 776 KB        -> k=8 = 6,06 MiB
    #
    # Ảnh giao thông gấp 3,3 lần batch 1: 1080p đầy chi tiết tần số cao (mặt
    # đường, nhiều vật nhỏ, nhiễu đêm) nên JPEG nén kém. Để 4,5 MB như trước là
    # batch nhóm N tự cắt xuống ~5 ảnh, và `BATCH_SIZE = 8` thành con số nói dối.
    #
    # 7,0 MB chừa ~1 MB biên dưới trần 8 MB cho prompt + khung JSON. `_chia_batch`
    # cắt theo byte nên nó TỰ điều tiết: ảnh nhẹ thì đủ 8, ảnh nặng thì ít hơn.
    raise ValueError("BATCH_MAX_BYTES phải nằm trong [700_000, 7_500_000]")
if DUONG_GOI == "batch" and BACKOFF_BASE < 20:
    # Chỉ ràng cho đường BATCH. Lùi ngắn hơn thời gian một batch là quay lại
    # đúng lúc suất đồng thời vẫn bận. Đo: nền 2s -> 8/18 batch chết ở 4 luồng;
    # nền 30s -> 0/18 ở 2 luồng. Đường `chat` thì request chỉ ~8s nên nền 2s là
    # đúng — ràng nó theo luật của batch là bắt nó chờ vô cớ.
    raise ValueError("BACKOFF_BASE phải ≥ 20 khi DUONG_GOI == 'batch'")
if DUONG_GOI == "chat" and MAX_WORKERS > 28:
    # 28 là mức đã chạy 6–9 giờ sạch đêm 21-09. 30 luồng thì cổng trả
    # `EOF occurred in violation of protocol` hàng loạt — đứt ở tầng TLS.
    raise ValueError("MAX_WORKERS > 28 ở đường chat: 30 luồng đã đo ra đứt TLS")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUNNER={RUNNER} · RUN_SCOPE={RUN_SCOPE} · model={MODEL_NAME} · "
      f"prompt={PROMPT_VARIANT} · batch={BATCH_SIZE} · max_bytes={BATCH_MAX_BYTES:,}")

In [ ]:
import subprocess, sys, os

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

# Repo là PRIVATE — `git clone` trần sẽ hỏi mật khẩu rồi TREO notebook, không
# phải báo lỗi. `GIT_ASKPASS` trả token qua stdin, `GIT_TERMINAL_PROMPT=0` biến
# mọi lần hỏi còn sót thành lỗi ngay thay vì treo. Cùng cách nb33 đã dùng.
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; '
        '*) printf %s "$GITHUB_TOKEN" ;; esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": secrets.get_secret("GITHUB_TOKEN"),
                "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL,
                    str(REPO_DIR)], check=True, env=env)
sys.path.insert(0, str(REPO_DIR / "src"))

os.environ["TRAMMAY_API_KEY"] = secrets.get_secret(TRAMMAY_SECRET)
for name in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"):
    os.environ[name] = secrets.get_secret(name)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modal"], check=True)

from mfe.contracts.ids import build_keyframe_id
from mfe.offline.bundle import BundleLayout
from mfe.offline.caption_prompt import PROMPT_VARIANTS
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.trammay import QuotaError, TrammayClient, TrammayError, available

from mfe.offline.shot_record_prompt import PROMPT_BAN_GHI_SHOT

PROMPT = PROMPT_BAN_GHI_SHOT
PROMPT_SUFFIX = ""    # KHÔNG dán `len180` — xem docstring `_va_prompt`

ok, vi_sao = available()
if not ok:
    raise RuntimeError(vi_sao)

# Chữ ký là `(volume_name, *, create_if_missing, version, dotenv_path)` — tên
# volume đi VỊ TRÍ, và KHÔNG có tham số `local_root`. Gọi `volume=...` ném
# TypeError ngay ở ô này.
RESEARCH = ModalVolumeTransport(RESEARCH_VOLUME, create_if_missing=False)
BTC = ModalVolumeTransport(BTC_VOLUME, create_if_missing=False)
BTC_LAYOUT = BundleLayout(BTC_BUNDLE)
CACHE_ANH.mkdir(parents=True, exist_ok=True)

# `backoff_base=30`, KHÔNG phải mặc định 2,0.
#
# Trần đồng thời của cổng đếm request ĐANG BAY, và một batch 8 ảnh chạy 23–47s.
# Lùi 2s/4s/8s là thử lại vào đúng lúc các suất vẫn chưa trống — cả bốn lần đều
# trượt. Đo 22-09 trên cùng 140 khung, 2 luồng: nền 2s -> 1/18 batch chết; nền
# 30s -> **0/18**, 140/140 caption.
client = TrammayClient.from_env(
    timeout=REQUEST_TIMEOUT, max_retries=MAX_RETRIES, backoff_base=BACKOFF_BASE
)
client = client.__class__(**{**client.__dict__, "model": MODEL_NAME})
print(f"prompt {PROMPT_VARIANT}: {len(PROMPT)} ký tự · batch={BATCH_SIZE} · client sẵn sàng")

## Chia việc — theo VIDEO, không theo keyframe

Chia theo video để mỗi worker tự đủ: một video là một đơn vị marker, và hai worker không bao giờ ghi cùng một file. Phép chia là **xác định** (băm tên video), nên chạy lại ở máy khác vẫn ra đúng cùng phần việc — không cần bảng phân công lưu ở đâu cả.

**Danh sách video đọc từ chính volume BTC**, không từ dataset Kaggle. Phép băm chia việc chạy trên danh sách đó, nên sáu worker **bắt buộc** thấy cùng một tập — mà đó chính là điều kiện để không video nào bị bỏ sót.


In [ ]:
import csv, hashlib, json

def _liet_ke_video() -> list:
    """Video ĐÃ CÓ caption keyframe — đó là đầu vào của tầng này."""
    ra = []
    for name in RESEARCH.listdir(f"{CAPTION_ROOT}/parts/captions"):
        ten = name.rsplit("/", 1)[-1]
        if ten.endswith(".json"):
            ra.append(ten[:-5])
    return sorted(ra)

VIDEOS = _liet_ke_video()
print(f"volume BTC: {len(VIDEOS)} video")
# KHÔNG ném khi thiếu: caption batch 1 còn đang sinh, nên danh sách lớn dần.
# Nhưng PHẢI in ra, vì chia việc tính trên chính danh sách này.
if len(VIDEOS) != EXPECTED_VIDEOS:
    print(f"⚠ {len(VIDEOS)}/{EXPECTED_VIDEOS} video có caption — "
          f"còn {EXPECTED_VIDEOS - len(VIDEOS)} video CHƯA sinh xong caption.\n"
          "   Shard tính trên danh sách HIỆN TẠI, nên chạy lại sau khi\n"
          "   caption xong sẽ thấy thêm video mới — đó là hành vi ĐÚNG.")

def shard_cua(video_id: str) -> str:
    """Băm xác định -> tên worker. Đổi hàm này là đổi toàn bộ phân công."""
    h = hashlib.sha256(video_id.encode()).hexdigest()
    return RUNNERS[int(h[:8], 16) % len(RUNNERS)]

PHAN_CONG = {r: [v for v in VIDEOS if shard_cua(v) == r] for r in RUNNERS}
for r in RUNNERS:
    print(f"  {r}: {len(PHAN_CONG[r]):>3} video")
CUA_TOI = PHAN_CONG[RUNNER] if RUN_SCOPE != "PREFLIGHT" else PHAN_CONG[RUNNER][:1]
print(f"\n{RUNNER} xử lý {len(CUA_TOI)} video trong lượt này")

## Shot + caption — `shot_id` lấy NGUYÊN VĂN từ gói audit\n\nGói dùng **4 chữ số** (`L21_V001#s0000`) còn `ids.SHOT_DIGITS` khai 2. `_SHOT_RE` parse được cả hai nên **không có gì nổ** — hai bên chỉ lặng lẽ sinh hai khoá cho cùng một shot. Ở đây bám theo DỮ LIỆU và không bao giờ dựng lại `shot_id` bằng `build_shot_id`.\n

In [ ]:
import re, zipfile
from collections import defaultdict

# Gói audit TransNetV2 (97.597 shot). Tải MỘT lần cho cả phiên.
_zip_cuc_bo = WORK_DIR / "shot-audit.zip"
if not _zip_cuc_bo.is_file():
    RESEARCH.get(SHOT_ZIP_REMOTE, _zip_cuc_bo)

def _nap_shot() -> dict:
    """`video_id -> [shot]`, mỗi shot sắp theo `start_frame`."""
    ra = defaultdict(list)
    with zipfile.ZipFile(_zip_cuc_bo) as z:
        for dong in z.read("shot-audit.jsonl").decode("utf-8").splitlines():
            d = json.loads(dong)
            ra[d["video_id"]].append(d)
    for v in ra:
        ra[v].sort(key=lambda x: x["start_frame"])
    return dict(ra)

SHOT = _nap_shot()
print(f"gói audit: {sum(len(v) for v in SHOT.values()):,} shot / {len(SHOT)} video")
if sum(len(v) for v in SHOT.values()) != EXPECTED_SHOTS:
    raise RuntimeError("gói shot không khớp EXPECTED_SHOTS — DỪNG")

_FRAME_RE = re.compile(r"#(\d{6})$")

def gan_caption(video_id: str, caps: dict) -> dict:
    """`shot_id -> [(frame_idx, caption)]`.

    KHÔNG cần `map-keyframes`: `keyframe_id = {video}#{frame_idx:06d}` đã mang
    sẵn `frame_idx`, nên chỉ việc giao nó với `[start_frame, end_frame]`.
    Khung rơi vào khe giữa hai shot thì BỎ, không gán bừa sang shot gần nhất —
    gán bừa là làm hỏng chính thứ tầng này bán: ranh giới cú máy.
    """
    theo_khung = []
    for kid, text in caps.items():
        m = _FRAME_RE.search(kid)
        if m:
            theo_khung.append((int(m.group(1)), text))
    theo_khung.sort()
    ra, i = {}, 0
    for s in SHOT.get(video_id, []):
        while i < len(theo_khung) and theo_khung[i][0] < s["start_frame"]:
            i += 1
        j = i
        trong = []
        while j < len(theo_khung) and theo_khung[j][0] <= s["end_frame"]:
            trong.append(theo_khung[j])
            j += 1
        i = j
        if trong:
            ra[s["shot_id"]] = trong
    return ra

def _thua_keyframe(ds: list) -> list:
    """Lấy mẫu THƯA khi shot có quá nhiều keyframe — giữ đầu và cuối."""
    if len(ds) <= MAX_KF_TRONG_SHOT:
        return ds
    buoc = (len(ds) - 1) / (MAX_KF_TRONG_SHOT - 1)
    return [ds[round(k * buoc)] for k in range(MAX_KF_TRONG_SHOT)]

def chia_cum(video_id: str, gan: dict) -> list:
    """`[[shot…]]` — cụm shot LIỀN NHAU, cắt theo ngân sách token."""
    cum, hien, tok = [], [], 0
    for s in SHOT.get(video_id, []):
        ds = gan.get(s["shot_id"])
        if not ds:
            continue                      # shot C0: không có keyframe nào
        ds = _thua_keyframe(ds)
        n = sum(len(c) for _, c in ds) // 4 + 40
        if hien and (len(hien) >= CUM_SHOT_TOI_DA or tok + n > CUM_TOKEN_TOI_DA):
            cum.append(hien)
            hien, tok = [], 0
        hien.append({"shot_id": s["shot_id"],
                     "duration_seconds": s["duration_seconds"],
                     "captions": [c for _, c in ds]})
        tok += n
    if hien:
        cum.append(hien)
    return cum

In [ ]:
import time
from datetime import datetime, timezone

from mfe.offline.shot_record_prompt import dung_dau_vao
from mfe.offline.caption_prompt import loc_khoi_ocr

def sha256_file(p) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for khoi in iter(lambda: fh.read(1 << 20), b""):
            h.update(khoi)
    return h.hexdigest()

def duong_remote(video_id: str) -> dict:
    return {
        "records": f"{OUTPUT_ROOT}/parts/shot_records/{video_id}.json",
        "marker": f"{OUTPUT_ROOT}/markers/{RUNNER}/{video_id}.json",
        # Checkpoint ở thư mục KHÁC: "có file trong parts/" phải tiếp tục có
        # nghĩa là "video đã xong", vì AGGREGATE đọc đúng chỗ đó.
        "checkpoint": f"{OUTPUT_ROOT}/checkpoints/{RUNNER}/{video_id}.json",
    }

def marker_hop_le(video_id: str):
    """Marker là điểm chốt, không phải file — part đẩy dở vẫn tồn tại trên volume.

    `ModalVolumeTransport` KHÔNG có `.read()`; phải `get()` về file rồi đọc.
    (Bắt được bằng `scratchpad/kiem_api_nb34.py` — đúng lớp lỗi mà script đó
    sinh ra để bắt: tên có thật, chữ ký thì không.)
    """
    paths = duong_remote(video_id)
    if not RESEARCH.exists(paths["marker"]):
        return None
    local = WORK_DIR / "marker-check" / f"{video_id}.json"
    local.parent.mkdir(parents=True, exist_ok=True)
    try:
        RESEARCH.get(paths["marker"], local)
        m = json.loads(local.read_text(encoding="utf-8"))
    except Exception:
        return None
    if m.get("status") != "COMPLETE" or m.get("runner") != RUNNER:
        return None
    if m.get("prompt_variant") != PROMPT_VARIANT or m.get("model") != MODEL_NAME:
        return None   # đổi prompt/model là phải sinh lại, không dùng lại bản cũ
    if not RESEARCH.exists(paths["records"]):
        return None
    if m.get("records_sha256") != RESEARCH.sha256(paths["records"]):
        return None      # marker nói xong mà file đã đổi -> KHÔNG tin marker
    return m

def _tach_json(raw: str):
    """Model hay bọc JSON trong ```json … ```. Bóc ra, đừng chết vì cái vỏ."""
    s = raw.strip()
    if "```" in s:
        for k in s.split("```"):
            k = k.removeprefix("json").strip()
            if k.startswith("{"):
                s = k
                break
    i, j = s.find("{"), s.rfind("}")
    if i < 0 or j <= i:
        return None
    try:
        return json.loads(s[i:j + 1])
    except json.JSONDecodeError:
        return None

def mot_cum(cum: list, bo_nho_truoc):
    """Một cụm shot -> `(dict shot_id->bản ghi, memory, lỗi|None)`.

    Đối chiếu `shot_id` trả về với `shot_id` gửi đi và VỨT cả cụm khi lệch:
    một bản ghi gắn nhầm shot là sai địa chỉ thời gian, và không phép kiểm nội
    dung nào nhìn ra — cùng lớp bẫy với `chat_batch` kiểm tập id.
    """
    mong = [s["shot_id"] for s in cum]
    try:
        raw = client.chat(PROMPT + "\n\n" + dung_dau_vao(cum, bo_nho_truoc),
                          max_tokens=MAX_TOKENS_CHAT, temperature=0.0)
    except QuotaError:
        raise
    except TrammayError as e:
        return {}, None, f"gọi hỏng: {str(e)[:120]}"
    d = _tach_json(raw)
    if not isinstance(d, dict) or not isinstance(d.get("shots"), list):
        return {}, None, "không parse được JSON"
    ra = {}
    for b in d["shots"]:
        sid = b.get("shot_id")
        if sid in mong and sid not in ra:
            # `onscreen_text` đi qua ĐÚNG bộ lọc đã dùng cho caption batch 2:
            # đo 24-09 thấy model vẫn nhặt mảnh logo `60 giây` vào đây.
            b["onscreen_text"] = [x for x in (
                loc_khoi_ocr("\n".join(b.get("onscreen_text") or [])).splitlines()) if x]
            b["program"] = d.get("program")
            ra[sid] = b
    if set(ra) != set(mong):
        thieu = [x for x in mong if x not in ra]
        return {}, None, f"lệch shot_id (thiếu {thieu[:3]})"
    return ra, d.get("memory"), None

def xu_ly_video(video_id: str, publish: bool) -> dict:
    local = WORK_DIR / "parts" / video_id
    local.mkdir(parents=True, exist_ok=True)
    file_rec = local / "records.json"
    remote = duong_remote(video_id)
    if not file_rec.is_file() and publish and RESEARCH.exists(remote["checkpoint"]):
        try:
            RESEARCH.get(remote["checkpoint"], file_rec)
            if not isinstance(json.loads(file_rec.read_text(encoding="utf-8")), dict):
                file_rec.unlink(missing_ok=True)
        except Exception:
            file_rec.unlink(missing_ok=True)
    recs = json.loads(file_rec.read_text(encoding="utf-8")) if file_rec.is_file() else {}

    cap_cuc_bo = WORK_DIR / "caps" / f"{video_id}.json"
    cap_cuc_bo.parent.mkdir(parents=True, exist_ok=True)
    if not cap_cuc_bo.is_file():
        RESEARCH.get(f"{CAPTION_ROOT}/parts/captions/{video_id}.json", cap_cuc_bo)
    caps = json.loads(cap_cuc_bo.read_text(encoding="utf-8"))
    gan = gan_caption(video_id, caps)
    cum_tat_ca = chia_cum(video_id, gan)
    con_lai = [c for c in cum_tat_ca
               if any(s["shot_id"] not in recs for s in c)]

    t0, hong, bo_nho = time.time(), 0, None
    for k, cum in enumerate(con_lai, 1):
        phan, mem, loi = mot_cum(cum, bo_nho)
        if loi:
            hong += 1
            print(f"  cụm {k}/{len(con_lai)} hỏng: {loi}")
        else:
            recs.update(phan)
            bo_nho = mem      # hồi quy ở mức CỤM, không phải mức shot
        if k % 5 == 0:
            file_rec.write_text(json.dumps(recs, ensure_ascii=False), encoding="utf-8")
            if publish:
                RESEARCH.put(file_rec, remote["checkpoint"])
    file_rec.write_text(json.dumps(recs, ensure_ascii=False), encoding="utf-8")

    n_shot_co_cap = len(gan)
    bao_cao = {"video_id": video_id, "shot": len(SHOT.get(video_id, [])),
               "shot_co_caption": n_shot_co_cap, "ban_ghi": len(recs),
               "cum": len(cum_tat_ca), "hong": hong,
               "giay": round(time.time() - t0, 1)}
    if not publish:
        return bao_cao
    if len(recs) != n_shot_co_cap:
        bao_cao["status"] = "PARTIAL"
        return bao_cao
    if n_shot_co_cap == 0:
        # Video không có shot nào chứa keyframe. KHÔNG ghi marker COMPLETE:
        # `0 == 0` lọt qua phép kiểm PARTIAL và video biến mất khỏi mọi báo cáo
        # thiếu — đúng bẫy đã gặp với `M09_V026` ở nb35.
        bao_cao["status"] = "KHONG_CO_SHOT_CO_CAPTION"
        return bao_cao

    RESEARCH.put(file_rec, remote["records"])
    if RESEARCH.sha256(remote["records"]) != sha256_file(file_rec):
        raise RuntimeError(f"{video_id}: records hash mismatch sau khi đẩy")
    marker = {"schema_version": 1, "status": "COMPLETE", "runner": RUNNER,
              "video_id": video_id, "shots": len(recs), "model": MODEL_NAME,
              "prompt_variant": PROMPT_VARIANT, "duong_goi": DUONG_GOI,
              "cum": len(cum_tat_ca), "max_kf_trong_shot": MAX_KF_TRONG_SHOT,
              "records_sha256": sha256_file(file_rec),
              "heldout_accessed": False, "btc_mutated": False,
              "completed_at_utc": datetime.now(timezone.utc).isoformat()}
    tmp = local / "marker.json"
    tmp.write_text(json.dumps(marker, ensure_ascii=False, indent=1), encoding="utf-8")
    RESEARCH.put(tmp, remote["marker"])     # marker ghi SAU CÙNG
    bao_cao["status"] = "COMPLETE"
    return bao_cao

In [ ]:
def cho_het_quota() -> bool:
    """Ngủ tới khi cửa sổ hạn mức thượng nguồn mở lại. `True` = đi tiếp được.

    Vì sao NGỦ chứ không thoát: cửa sổ hạn mức là 5 tiếng còn phiên Kaggle sống
    12 tiếng, nên chờ hết cửa sổ rồi làm tiếp vẫn còn dư thời gian — trong khi
    thoát ra là phải có người ngồi canh để bấm chạy lại. Đo 24-09: một worker
    dừng lúc 4,95 giờ rồi nằm không suốt đêm.

    Vì sao THĂM DÒ chứ không ngủ đúng 5 tiếng: không ai biết cửa sổ bắt đầu lúc
    nào. Chạm trần ở phút thứ 10 của cửa sổ thì phải chờ 4h50; chạm ở phút 290
    thì chỉ chờ 10 phút. Ngủ cứng 5 tiếng là **luôn** chọn trường hợp xấu nhất.

    Thăm dò bằng một lượt gọi KHÔNG ẢNH (vài chục token, và quan trọng hơn là
    **một** request), nên 5 tiếng chờ tốn tối đa ~30 request trên 1.000 của cửa
    sổ. Lỗi KHÁC quota (5xx, timeout) không nói gì về hạn mức — coi như chưa
    biết và chờ tiếp, chứ đừng kết luận đã mở.

    Phiên có chết giữa lúc chờ cũng không mất gì: marker và checkpoint đã nằm
    trên volume, chạy lại là tiếp đúng chỗ dở.
    """
    t0 = time.time()
    tran = CHO_QUOTA_PHUT * 60
    while time.time() - t0 < tran:
        time.sleep(NHIP_THAM_DO_PHUT * 60)
        phut = (time.time() - t0) / 60
        try:
            client.chat("ping", max_tokens=8)
            print(f"   ✅ hạn mức đã mở lại sau {phut:.0f} phút chờ", flush=True)
            return True
        except QuotaError:
            print(f"   … vẫn hết hạn mức ({phut:.0f}/{CHO_QUOTA_PHUT} phút)", flush=True)
        except TrammayError as e:
            # Không phải tín hiệu về quota. Chờ tiếp thay vì đoán bừa.
            print(f"   … thăm dò lỗi khác ({phut:.0f} phút): {str(e)[:70]}", flush=True)
    return False

if RUN_SCOPE in {"PREFLIGHT", "SHARD"}:
    xong_roi = ([v for v in CUA_TOI if marker_hop_le(v)]
                if RUN_SCOPE == "SHARD" else [])
    con_lai = [v for v in CUA_TOI if v not in set(xong_roi)]
    print(f"{len(xong_roi)} video đã COMPLETE · còn {len(con_lai)}")

    bao_caos, that_bai = [], []
    dung_han = False
    for i, video_id in enumerate(con_lai, 1):
        if dung_han:
            break
        # Vòng trong CHỈ lặp khi chờ được hạn mức mở lại — để làm tiếp ĐÚNG
        # video đang dở, không bỏ nó lại rồi nhảy sang video sau.
        while True:
            try:
                r = xu_ly_video(video_id, publish=RUN_SCOPE == "SHARD")
                bao_caos.append(r)
                print(f"[{i}/{len(con_lai)}] {video_id}: {r['caption']}/{r['khung']} "
                      f"· hỏng {r['hong']} · {r['giay']}s · {r.get('status', 'PREFLIGHT')}")
            except QuotaError as e:
                # KHÔNG quét tiếp danh sách khi hạn mức đã cạn: làm thế chỉ tạo
                # ra một loạt video PARTIAL và đốt hết phiên (đo 23-09: 1.419,9s
                # cho MỘT video mà thu về 0 caption). Ngủ chờ cửa sổ reset rồi
                # làm tiếp chính video này.
                print(f"\n⛔ HẾT HẠN MỨC ở video {i}/{len(con_lai)} ({video_id})")
                print(f"   {str(e)[:200]}")
                print(f"   Chờ cửa sổ hạn mức reset, thăm dò mỗi "
                      f"{NHIP_THAM_DO_PHUT} phút…", flush=True)
                if cho_het_quota():
                    continue          # làm lại CHÍNH video này
                print(f"   Chờ quá {CHO_QUOTA_PHUT} phút mà chưa mở. DỪNG lượt chạy;")
                print("   phần đã xong giữ nguyên marker, chạy lại là tiếp chỗ dở.")
                dung_han = True
            except Exception as e:
                that_bai.append({"video_id": video_id, "loi": repr(e)})
                print(f"[{i}/{len(con_lai)}] {video_id}: THẤT BẠI {e!r}")
            break

    tong_hong = sum(r["hong"] for r in bao_caos)
    tong_cap = sum(r["caption"] for r in bao_caos)
    print(f"\nTổng: {tong_cap:,} caption · {tong_hong} khung hỏng "
          f"({tong_hong / max(1, tong_cap + tong_hong) * 100:.1f}%)")
    if tong_hong / max(1, tong_cap + tong_hong) > 0.02:
        print("⚠ >2% hỏng — xem dòng `hỏng` ở trên để biết vì sao:\n"
              "   429 quota -> DỪNG, đợi cửa sổ hạn mức reset (5 tiếng)\n"
              "   HTTP 413  -> hạ BATCH_MAX_BYTES\n"
              "   max_tokens -> nới MAX_TOKENS\n"
              "   EOF/TLS    -> hạ MAX_WORKERS")

In [ ]:
if RUN_SCOPE == "AGGREGATE":
    if not CONFIRM_AGGREGATE:
        raise RuntimeError("đặt CONFIRM_AGGREGATE = True sau khi cả 6 shard COMPLETE")
    xong = [v for v in VIDEOS if marker_hop_le(v)]
    print(f"{len(xong)}/{len(VIDEOS)} video có marker hợp lệ")
    if len(xong) != len(VIDEOS):
        thieu = [v for v in VIDEOS if v not in set(xong)]
        raise RuntimeError(f"còn {len(thieu)} video chưa COMPLETE: {thieu[:5]}")

    gop = {}
    for i, video_id in enumerate(xong, 1):
        dich = WORK_DIR / "gop" / f"{video_id}.json"
        dich.parent.mkdir(parents=True, exist_ok=True)
        RESEARCH.get(f"{OUTPUT_ROOT}/parts/shot_records/{video_id}.json", dich)
        gop.update(json.loads(dich.read_text(encoding="utf-8")))
        if i % 100 == 0:
            print(f"  gộp {i}/{len(xong)}")
    print(f"tổng {len(gop):,} bản ghi shot")

    # Hai artifact, hai mục đích:
    #   `shot_records.json` giữ NGUYÊN cấu trúc, cho tầng cửa sổ + soi tay;
    #   `shot_text.json`    là chuỗi phẳng cho BM25/BGE-M3 — chỉ những trường
    #                       THẬT SỰ có nguồn, không có `link_to_previous` (nó là
    #                       quan hệ, không phải nội dung để tìm).
    phang, rong = {}, 0
    for sid, b in gop.items():
        phan = [b.get("event") or ""]
        phan += list(b.get("entities") or [])
        if b.get("state_change"):
            phan.append(b["state_change"])
        phan += list(b.get("onscreen_text") or [])
        t = " . ".join(x for x in phan if x).strip()
        if t:
            phang[sid] = t
        else:
            rong += 1
    print(f"phẳng: {len(phang):,} chuỗi · {rong} bản ghi rỗng")

    ra = WORK_DIR / "shot_records.json"
    ra.write_text(json.dumps(gop, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(ra, f"{OUTPUT_ROOT}/shot_records.json")
    ra2 = WORK_DIR / "shot_text.json"
    ra2.write_text(json.dumps(phang, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(ra2, f"{OUTPUT_ROOT}/shot_text.json")
    print("đã đẩy shot_records.json + shot_text.json")

## Sau khi AGGREGATE\n\n`shot_records.json` giữ nguyên cấu trúc (cho tầng cửa sổ + soi tay); `shot_text.json` là chuỗi phẳng cho BM25/BGE-M3.\n\n⚠ **Chốt 4 trước khi đưa vào kho**: lấy 50 shot, người đọc đối chiếu `event` và `state_change` với video. `state_change` sai là dạng bịa nguy hiểm nhất ở tầng này — nó điền vào đúng ô phục vụ truy vấn T4 một biến đổi chưa từng xảy ra.\n